# Explore, Visualize, and Prepare the Data

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from pandas.plotting import scatter_matrix

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.preprocessing.prepare_the_data import (
    get_event_feature_groups,
    prepare_weighted_event_data,
)

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
weighted_path = event_dir / f"aapl_weighted_events_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
prepared_path = event_dir / f"aapl_model_events_{period}.parquet"

weighted_events = pd.read_parquet(weighted_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")
close = dollar_bars.set_index("end")["close"].astype(float)

development = weighted_events.loc[
    weighted_events["partition"].eq("development")
].copy()
exploration_groups = get_event_feature_groups(development)
exploration_features = [
    column for columns in exploration_groups.values() for column in columns
]

## Visualizing Data

- **Purpose:** Visualize representative fractional-price, sentiment, momentum, trend, and volatility features.
- **Settings:** `feature_count=6`; the displayed features are fixed before inspection.
- **Data:** Plot six representative features from the development partition.
- **Decision:** Treat the plots as diagnostics that do not change the feature schema.

In [2]:
representative_features = [
    "fractionally_differenced_log_close",
    "mean_sentiment_score",
    "McClellan Oscillator",
    "Relative Strength Index",
    "Simple Moving Average (SMA)",
    "Average True Range",
]
development.set_index("event_start")[representative_features].plot(
    subplots=True,
    figsize=(12, 12),
    grid=True,
)
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_61257/1644777589.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Look for Correlations

- **Purpose:** Measure Pearson correlations among the direction label and all development features.
- **Settings:** `feature_count=53`; `top_k=3` by absolute target correlation.
- **Data:** Use the direction label and all 53 features from the development partition.
- **Decision:** Keep the ranking descriptive rather than applying feature selection.

In [3]:
correlation_matrix = development[
    [*exploration_features, "direction_label"]
].corr(numeric_only=True)
target_correlations = correlation_matrix["direction_label"].drop(
    "direction_label"
)
target_correlations = target_correlations.reindex(
    target_correlations.abs().sort_values(ascending=False).index
)
display(
    target_correlations.rename("direction_label_correlation").to_frame()
)

scatter_features = target_correlations.head(3).index.tolist()
scatter_matrix(
    development[["direction_label", *scatter_features]],
    figsize=(12, 12),
    diagonal="hist",
    alpha=0.35,
)
plt.tight_layout()
plt.show()

,direction_label_correlation
Commodity Channel Index,-0.134478
Stochastic %D,0.128011
True Range,-0.117128
Detrended Price Oscillator,0.105183
Average True Range,-0.089940
Aroon Indicator Down,0.087972
Balance of Power,-0.081342
Relative Vigor Index,-0.062661
Williams %R,0.054087
Stochastic %K,0.054087


/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_61257/81352967.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Clean the Data

- **Purpose:** Enforce the predeclared completeness rule on the fixed modeling schema.
- **Settings:** `feature_count=53`; `imputation=None`; remove rows containing any missing or infinite feature value.
- **Data:** Clean the fixed 53-feature schema in both inline partitions and refresh event weights when rows are removed.
- **Decision:** Apply the same rule to both partitions, record removals, and recompute weights within each affected partition.

In [4]:
prepared_events, cleaning_report = prepare_weighted_event_data(
    weighted_events,
    close,
)

event_dir.mkdir(parents=True, exist_ok=True)
prepared_events.to_parquet(prepared_path, index=False)

display(
    cleaning_report.groupby("feature_group")[
        ["missing_values", "infinite_values", "invalid_rows"]
    ].sum()
)
display(
    prepared_events["partition"]
    .value_counts()
    .rename("events")
    .to_frame()
)
print(prepared_path)

,missing_values,infinite_values,invalid_rows
feature_group,,,
fractional_price,0,0,0
sentiment,0,0,0
technical,0,0,0


,events
partition,
development,176
holdout,22


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_model_events_2025-01-01_2025-12-31.parquet
